# Home Credit Baseline - Binary Classification Model

This notebook builds a binary classification model using traditional, boosted and deep learning methodes to predict loan defaults based on home credit data and test its stability over time.

**Source:** https://www.kaggle.com/code/greysky/home-credit-baseline

## Step 1: Import Libraries and Set Paths

Import required libraries for data processing (polars, numpy), visualization (matplotlib, seaborn), and machine learning (scikit-learn, LightGBM).
Define paths to training, testing, and sample data directories.


In [ ]:
import gc
from glob import glob

import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import BaseEstimator, ClassifierMixin

import lightgbm as lgb


TRAIN_DIR = "data/train"
TEST_DIR = "data/test"
SAMPLE_DIR = "data/sample"

## Step 2: Define Helper Classes

### Pipeline Class
Handles data preprocessing steps:
- **set_table_dtypes**: Cast columns to appropriate data types (Int32, Float64, Date, String)
- **handle_dates**: Convert date columns to days relative to decision date
- **filter_cols**: Remove columns with >95% missing values or low variance (categorical with 1 or >200 unique values)

In [ ]:
class Pipeline:
    @staticmethod
    def set_table_dtypes(df):
        for col in df.columns:
            if col in ["case_id", "WEEK_NUM", "num_group1", "num_group2"]:
                df = df.with_columns(pl.col(col).cast(pl.Int32))
            elif col in ["date_decision"]:
                df = df.with_columns(pl.col(col).cast(pl.Date))
            elif col[-1] in ("P", "A"):
                df = df.with_columns(pl.col(col).cast(pl.Float64))
            elif col[-1] in ("M",):
                df = df.with_columns(pl.col(col).cast(pl.String))
            elif col[-1] in ("D",):
                df = df.with_columns(pl.col(col).cast(pl.Date))

        return df

    @staticmethod
    def handle_dates(df):
        for col in df.columns:
            if col[-1] in ("D",):
                df = df.with_columns(pl.col(col) - pl.col("date_decision"))
                df = df.with_columns(pl.col(col).dt.total_days())
                df = df.with_columns(pl.col(col).cast(pl.Float32))

        df = df.drop("date_decision", "MONTH")

        return df

    @staticmethod
    def filter_cols(df):
        for col in df.columns:
            if col not in ["target", "case_id", "WEEK_NUM"]:
                isnull = df[col].is_null().mean()

                if isnull > 0.95:
                    df = df.drop(col)

        for col in df.columns:
            if (col not in ["target", "case_id", "WEEK_NUM"]) & (df[col].dtype == pl.String):
                freq = df[col].n_unique()

                if (freq == 1) | (freq > 200):
                    df = df.drop(col)

        return df

### Aggregator Class
Aggregates features by case_id at different depth levels. Creates maximum values for:
- Numeric features (columns ending in "P" or "A")
- Date features (columns ending in "D")
- String features (columns ending in "M")
- Other categorical features (columns ending in "T" or "L")
- Group count columns


In [ ]:
class Aggregator:
    @staticmethod
    def num_expr(df):
        cols = [col for col in df.columns if col[-1] in ("P", "A")]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def date_expr(df):
        cols = [col for col in df.columns if col[-1] in ("D",)]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def str_expr(df):
        cols = [col for col in df.columns if col[-1] in ("M",)]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def other_expr(df):
        cols = [col for col in df.columns if col[-1] in ("T", "L")]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def count_expr(df):
        cols = [col for col in df.columns if "num_group" in col]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def get_exprs(df):
        exprs = Aggregator.num_expr(df) + \
            Aggregator.date_expr(df) + \
            Aggregator.str_expr(df) + \
            Aggregator.other_expr(df) + \
            Aggregator.count_expr(df)

        return exprs

## Step 3: Data Loading Functions

- **read_file**: Reads a single parquet file and applies type casting and aggregation
- **read_files**: Reads multiple parquet files matching a glob pattern, concatenates them, and removes duplicates


In [ ]:
def read_file(path, depth=None):
    df = pl.read_parquet(path)
    df = df.pipe(Pipeline.set_table_dtypes)

    if depth in [1, 2]:
        df = df.group_by("case_id").agg(Aggregator.get_exprs(df))

    return df


def read_files(regex_path, depth=None):
    chunks = []
    for path in glob(str(regex_path)):
        df = pl.read_parquet(path)
        df = df.pipe(Pipeline.set_table_dtypes)

        if depth in [1, 2]:
            df = df.group_by("case_id").agg(Aggregator.get_exprs(df))

        chunks.append(df)

    df = pl.concat(chunks, how="vertical_relaxed")
    df = df.unique(subset=["case_id"])

    return df

## Step 4: Feature Engineering

Combines base features with features from different data depths:
- Extracts month and weekday from decision date
- Joins multiple feature tables on case_id
- Converts date columns to relative days and handles missing values


In [ ]:
def feature_eng(df_base, depth_0, depth_1, depth_2):
    df_base = (
        df_base
        .with_columns(
            month_decision=pl.col("date_decision").dt.month(),
            weekday_decision=pl.col("date_decision").dt.weekday(),
        )
    )

    for i, df in enumerate(depth_0 + depth_1 + depth_2):
        df_base = df_base.join(df, how="left", on="case_id", suffix=f"_{i}")

    df_base = df_base.pipe(Pipeline.handle_dates)

    return df_base

## Step 5: Data Format Conversion

Converts polars DataFrame to pandas and converts object columns to categorical type for memory efficiency.


In [ ]:
def to_pandas(df_data, cat_cols=None):
    df_data = df_data.to_pandas()

    if cat_cols is None:
        cat_cols = list(df_data.select_dtypes("object").columns)

    df_data[cat_cols] = df_data[cat_cols].astype("category")

    return df_data, cat_cols

## Step 6: Load and Prepare Training and Testing Data

Loads all training and testing data files, performs feature engineering, and prepares the dataset. Training data includes base information plus features from different depths (static, applications, tax registry, credit bureau, etc.).


In [ ]:
data_store = {
    "df_base": read_file(f"{SAMPLE_DIR}/train_base_sampled.parquet"),
    "depth_0": [
        read_file(f"{SAMPLE_DIR}/train_static_cb_0_sampled.parquet"),
        read_files(f"{SAMPLE_DIR}/train_static_0_*.parquet"),
    ],
    "depth_1": [
        read_files(f"{SAMPLE_DIR}/train_applprev_1_*.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_tax_registry_a_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_tax_registry_b_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_tax_registry_c_1_sampled.parquet", 1),
        read_files(f"{SAMPLE_DIR}/train_credit_bureau_a_1_*.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_credit_bureau_b_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_other_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_person_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_deposit_1_sampled.parquet", 1),
        read_file(f"{SAMPLE_DIR}/train_debitcard_1_sampled.parquet", 1),
    ],
    "depth_2": [
        read_file(f"{SAMPLE_DIR}/train_credit_bureau_b_2_sampled.parquet", 2),
        read_files(f"{SAMPLE_DIR}/train_credit_bureau_a_2_*.parquet", 2),
    ]
}

df_train = feature_eng(**data_store)
print("train data shape:\t", df_train.shape)

In [ ]:
data_store = {
    "df_base": read_file(f"{TEST_DIR}/test_base.parquet"),
    "depth_0": [
        read_file(f"{TEST_DIR}/test_static_cb_0.parquet"),
        read_files(f"{TEST_DIR}/test_static_0_*.parquet"),
    ],
    "depth_1": [
        read_files(f"{TEST_DIR}/test_applprev_1_*.parquet", 1),
        read_file(f"{TEST_DIR}/test_tax_registry_a_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_tax_registry_b_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_tax_registry_c_1.parquet", 1),
        read_files(f"{TEST_DIR}/test_credit_bureau_a_1_*.parquet", 1),
        read_file(f"{TEST_DIR}/test_credit_bureau_b_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_other_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_person_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_deposit_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_debitcard_1.parquet", 1),
    ],
    "depth_2": [
        read_file(f"{TEST_DIR}/test_credit_bureau_b_2.parquet", 2),
        read_files(f"{TEST_DIR}/test_credit_bureau_a_2_*.parquet", 2),
    ]
}

df_test = feature_eng(**data_store)
print("test data shape:\t", df_test.shape)

## Step 7: Filter Features

Removes low-information columns from training data and keeps only the same features in test data.


In [ ]:
df_train = df_train.pipe(Pipeline.filter_cols)
df_test = df_test.select([col for col in df_train.columns if col != "target"])

print("train data shape:\t", df_train.shape)
print("test data shape:\t", df_test.shape)

## Step 8: Convert to Pandas Format

Converts both datasets to pandas DataFrames and converts categorical columns to category dtype for memory efficiency.


In [ ]:
df_train, cat_cols = to_pandas(df_train)
df_test, cat_cols = to_pandas(df_test, cat_cols)

del data_store
gc.collect()

## Step 9: Exploratory Data Analysis - Target Distribution Over Time

Visualizes how the target variable (loan default rate) changes across weeks to understand temporal patterns.


In [ ]:
sns.lineplot(
    data=df_train,
    x="WEEK_NUM",
    y="target",
)
plt.show()

In [ ]:
plt.scatter(data=df_train, x="dateofbirth_337D", y="target")
plt.xlabel("dateofbirth_337D")
plt.ylabel("target")

## Step 10: Model Training with Cross-Validation

Trains LightGBM models using Stratified Group K-Fold cross-validation:
- Respects week boundaries to prevent data leakage (shuffle=False keeps temporal order)
- Groups folds by week number to ensure complete weeks stay together
- Trains 5 models and combines them with voting ensemble
- Uses GPU acceleration for faster training
- Applies early stopping to prevent overfitting


In [ ]:
# # Prepare training features and target
# X = df_train.drop(columns=["target", "case_id", "WEEK_NUM"])
# y = df_train["target"]
# weeks = df_train["WEEK_NUM"]

# # Use Stratified Group K-Fold to maintain temporal order and respect week boundaries
# cv = StratifiedGroupKFold(n_splits=5, shuffle=False)

# # LightGBM parameters optimized for GPU training
# params = {
#     "boosting_type": "gbdt",  # Gradient Boosting Decision Tree
#     "objective": "binary",     # Binary classification task
#     "metric": "auc",           # Evaluation metric
#     "max_depth": 8,
#     "learning_rate": 0.05,
#     "n_estimators": 1000,
#     "colsample_bytree": 0.8,   # Subsample features per tree
#     "colsample_bynode": 0.8,   # Subsample features per split
#     "verbose": -1,
#     "random_state": 42,
#     "device": "gpu",           # Use GPU for acceleration -> TODO: fix bcs does not work lol
# }

# # Train models using cross-validation
# fitted_models = []

# for idx_train, idx_valid in cv.split(X, y, groups=weeks):
#     X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]
#     X_valid, y_valid = X.iloc[idx_valid], y.iloc[idx_valid]

#     # Train individual LightGBM model
#     model = lgb.LGBMClassifier(**params)
#     model.fit(
#         X_train, y_train,
#         eval_set=[(X_valid, y_valid)],
#         callbacks=[lgb.log_evaluation(100), lgb.early_stopping(100)]
#     )

#     fitted_models.append(model)

# # Create voting ensemble from all trained models
# model = VotingModel(fitted_models)